# CTC, attention encoder-decoder and hybrid CTC/attention, trained from scratch for Icelandic

Course baseline families **CTC encoder** and **attention-based encoder-decoder, including hybrid CTC/attention**.
All three reuse the RNN-T pipeline (`training/train_rnnt.py`): the same log-mel features, 8-layer Transformer encoder, data, batching and training budget, so the four neural families are directly comparable.

| Mode | Heads | Training loss | Final decoding |
|---|---|---|---|
| `ctc` | CTC | CTC | greedy CTC |
| `attention` | 4-layer Transformer decoder | label-smoothed cross-entropy | beam 5, length-normalized |
| `hybrid` | CTC + decoder | 0.3 × CTC + 0.7 × attention | beam 5, n-best rescored with 0.3 × CTC + 0.7 × attention |

Dev WER (greedy) selects each checkpoint; test is decoded once per model at the end. Settings: `Config` in `training/train_ctc_attention.py`, seed 2105.
For each mode the notebook first runs a 32-clip **memorization check** and stops if the loss does not fall.

Colab: runtime **T4 GPU**, then **Run all**; keep the tab open (~3–3.5 h for all three). To split the work, edit `MODES` in the first code cell.
Kaggle: GPU T4 x2, Internet On, then Save Version → Save & Run All.

In [ ]:
!nvidia-smi

## 1. Repository at fixed data and code commits

In [ ]:
MODES = ["ctc", "attention", "hybrid"]      # run order; edit to split across sessions

import os
WORK = "/kaggle/working" if "KAGGLE_KERNEL_RUN_TYPE" in os.environ else "/content"   # Colab also has a /kaggle folder
REPO_DIR = f"{WORK}/icelandic-asr-project3"

DATA_REPO = "https://github.com/RanimAlyaseen/icelandic-asr-project3.git"
DATA_COMMIT = "ef4d5156cb599348dd9dc55934ff391cc8c5ee5d"   # frozen v1 audio, manifests and references
CODE_REPO = "https://github.com/MuhammetGoksenin/icelandic-asr-project3.git"
CODE_COMMIT = "4912e53b396d82f17d9f71243d57a28fec353014"
CODE_PATHS = "baselines training tests/test_rnnt.py tests/test_ctc_attention.py requirements-modeling.txt"
assert len(CODE_COMMIT) == 40

!which git-lfs >/dev/null || (apt-get -qq update && apt-get -qq install -y git-lfs)
!git lfs install
if not os.path.exists(REPO_DIR):
    !GIT_LFS_SKIP_SMUDGE=1 git clone -q {DATA_REPO} {REPO_DIR}
%cd {REPO_DIR}
!GIT_LFS_SKIP_SMUDGE=1 git checkout -q {DATA_COMMIT}
!git lfs pull --include="data/cleaned/samromur-21.05-20h-v1/**" --exclude=""
!GIT_LFS_SKIP_SMUDGE=1 git fetch -q {CODE_REPO} {CODE_COMMIT}
!git checkout {CODE_COMMIT} -- {CODE_PATHS}
!echo "Data commit: $(git rev-parse HEAD)"; echo "Code commit: {CODE_COMMIT}"
!du -sh data/cleaned/samromur-21.05-20h-v1/*

## 2. Pinned packages and tests

In [ ]:
!pip install -q -r requirements-modeling.txt
!python -m unittest discover -s tests -p "test_ctc_attention.py" -v 2>&1 | tail -12

## 3. Train, select on dev, and decode test, one mode after another

**What happened in the reported run (Colab, one T4, 21–22 November 2026).** The cell below is the final version; the run reached the same results in three steps:

- **ctc**: memorization check passed (exact transcript), 60 epochs, best epoch 54, test decoded once.
- **attention**: the first version of this cell required the loss to fall below 10 % of its start, which label smoothing (0.1) prevents, so it stopped although the transcript was copied. With the check fixed, full training ran; the **plateau guard stopped it at epoch 16** (best greedy dev WER 173.4 %, repeated and inserted words). Its epoch-16 checkpoint was then decoded on test once, for reporting.
- **hybrid**: the memorization check cut loss from 5.45 to 0.52 (> 10×) but the greedy transcript at step 600 was not exact (`skrift myndlist handandavoru bras ég ég`). We continued because the loss fell and the plateau guard protects the full run. 60 epochs, best epoch 60, test decoded once.

The final cell below reproduces that sequence: it reports exact-transcript status instead of requiring it, and decodes test for a run stopped by the plateau guard.

In [ ]:
import subprocess, re, json

def run(args, allow_plateau_guard=False):
    p = subprocess.Popen(["python", "training/train_ctc_attention.py", *args],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    out = "".join(lines)
    stopped = "PLATEAU GUARD" in out
    assert p.wait() == 0 or (allow_plateau_guard and stopped), f"failed: {args}"
    return out

def memorization_report(out):
    """Label smoothing keeps attention loss above zero, so report the transcript and require a 70 % loss drop."""
    ref, hyp = re.findall(r"ref '(.*?)'\s+hyp '(.*?)'", out)[-1]
    losses = [float(m) for m in re.findall(r"loss\s+([0-9.]+)", out)]
    print(f"last step: loss {losses[0]:.3f} -> {losses[-1]:.3f}, exact transcript: {ref.strip() == hyp.strip()}")
    return losses[-1] < 0.3 * losses[0]

for mode in MODES:
    print(f"\n========== {mode}: memorization check ==========")
    out = run(["--mode", mode, "--overfit", "32", "--overfit-steps", "600"])
    assert memorization_report(out), f"{mode}: loss did not fall, stop and inspect"
    print(f"\n========== {mode}: full training ==========")
    run(["--mode", mode, "--train"], allow_plateau_guard=True)    # a stopped run is still decoded and reported
    print(f"\n========== {mode}: test (once) ==========")
    run(["--mode", mode, "--evaluate-test"])

## 4. Summary

In [ ]:
from IPython.display import Image, display
KEYS = {"ctc": "ctc_scratch", "attention": "attention_scratch", "hybrid": "hybrid_ctc_attention"}
for mode in MODES:
    key = KEYS[mode]
    s = json.load(open(f"results/{key}/training_summary.json"))
    t = json.load(open(f"results/{key}/model_info/{key}_test.json"))["scores_preliminary"]
    print(f"{key}: stopped by {s['stopped_by']}")
    print(f"{key}: best epoch {s['best_epoch']}, train {s['train_seconds']/60:.0f} min, "
          f"test WER {t['wer']*100:.2f}%  CER {t['cer']*100:.2f}%")
    display(Image(f"results/{key}/training_curves.png"))

## 5. Save results and weights

In [ ]:
keys = " ".join(KEYS[m] for m in MODES)
!pip freeze > {WORK}/pip_freeze.txt
for k in keys.split():
    !cp {WORK}/pip_freeze.txt results/{k}/pip_freeze.txt; cp output/ctc_vocab.json models/{k}/vocab.json
!cd results && zip -qr {WORK}/ctc_attention_results.zip {keys}
!cd models && zip -qr {WORK}/ctc_attention_models.zip {keys}
if WORK == "/kaggle/working":
    print("Download ctc_attention_results.zip and ctc_attention_models.zip from the Output panel.")
else:
    from google.colab import files
    files.download(f"{WORK}/ctc_attention_results.zip")
    files.download(f"{WORK}/ctc_attention_models.zip")